In [1]:
import optuna

%load_ext autoreload
%autoreload 2

In [ ]:

def sample_params(n, seed=0):
    sampler = optuna.samplers.TPESampler(seed=seed)
    study = optuna.create_study(sampler=sampler, direction="maximize")

    def objective(trial):
        trial.suggest_float("lr", 1e-5, 3e-1, log=True)
        trial.suggest_float("weight_decay", 1e-7, 5e-3, log=True)
        trial.suggest_int("batch_size", 16, 512, step=16)
        trial.suggest_float("momentum", 0.0, 0.9)
        return 0.0

    study.optimize(objective, n_trials=n)
    return [t.params for t in study.trials]

In [3]:
params = sample_params(5, 42)

[I 2026-10-05 17:24:02,032] A new study created in memory with name: no-name-0122ecc9-86ab-44c6-9bb4-9b7469c9074d
[I 2026-10-05 17:24:02,036] Trial 0 finished with value: 0.0 and parameters: {'lr': 0.0004751836912519933, 'weight_decay': 0.0029334453957883953, 'batch_size': 384, 'momentum': 0.5387926357773329}. Best is trial 0 with value: 0.0.
[I 2026-10-05 17:24:02,037] Trial 1 finished with value: 0.0 and parameters: {'lr': 4.9947570810682895e-05, 'weight_decay': 5.407712220288117e-07, 'batch_size': 32, 'momentum': 0.7795585311974417}. Best is trial 0 with value: 0.0.
[I 2026-10-05 17:24:02,037] Trial 2 finished with value: 0.0 and parameters: {'lr': 0.004912072810282769, 'weight_decay': 0.00021242802137208872, 'batch_size': 16, 'momentum': 0.8729188669457949}. Best is trial 0 with value: 0.0.
[I 2026-10-05 17:24:02,038] Trial 3 finished with value: 0.0 and parameters: {'lr': 0.05332693713874534, 'weight_decay': 9.948901058090725e-07, 'batch_size': 96, 'momentum': 0.16506405886809045}

In [10]:
import json

with open("experiments/five_teachers_params.json", "w") as f:
    json.dump(params, f, indent=2)

In [11]:
from pathlib import Path

import torch

from kd_vs_hpo.common.dataloader import build_dataloaders

train_loader, val_loader, test_loader, n_train, n_val, n_test= build_dataloaders(
    "cifar10", Path("./tmp/"), Path("./tmp/"), Path("data"), 0, 32, 4, 0.1, torch.device('cpu'), label_noise=0
)
train_labels = torch.cat([t[1] for t in train_loader])
val_labels = torch.cat([t[1] for t in val_loader])
test_labels = torch.cat([t[1] for t in test_loader])


In [21]:
train_loader_noise, val_loader_noise, test_loader_noise, n_train, n_val, n_test= build_dataloaders(
    "cifar10", Path("./tmp/"), Path("./tmp/"), Path("data"), 0, 32, 4, 0.1, torch.device('cpu'), label_noise=0.78
)

train_labels_noise = torch.cat([t[1] for t in train_loader_noise])
val_labels_noise = torch.cat([t[1] for t in val_loader_noise])
test_labels_noise = torch.cat([t[1] for t in test_loader_noise])

In [22]:
(test_labels == test_labels_noise).all()

tensor(True)

In [23]:
(train_labels != train_labels_noise).to(float).mean(), (val_labels != val_labels_noise).to(float).mean()

(tensor(0.7800, dtype=torch.float64), tensor(0.7800, dtype=torch.float64))

In [24]:
torch.bincount(train_labels)

tensor([4489, 4490, 4494, 4491, 4498, 4528, 4500, 4530, 4500, 4480])

In [25]:
torch.bincount(test_labels)

tensor([1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000, 1000])

In [26]:
torch.bincount(train_labels_noise)

tensor([4501, 4477, 4616, 4483, 4488, 4491, 4430, 4465, 4532, 4517])